# Energy Transfer Analysis
This notebook demonstrates the energy-transfer analysis target, including the heat-surplus/deficit table, graph-ready energy transfer diagram, and base-target selection for site and local scopes.

**Support notice:** `PinchProblem` and `PinchWorkspace` are public package-root workflows. Concrete owner modules used by this advanced notebook remain unsupported; `from OpenPinch.main import pinch_analysis_service` is the strict integration contract.


## Case context
The packaged `pulp_mill.json` case has a real zone tree, so it can show the Total Site energy-transfer view and a local direct-integration view without changing input files.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

from OpenPinch import PinchProblem
from OpenPinch.resources import copy_sample_case


In [ ]:
working_dir = TemporaryDirectory()
local_case_path = copy_sample_case(
    "pulp_mill.json",
    Path(working_dir.name) / "pulp_mill.json",
)

local_case_path


## Total Site energy transfer
For an aggregate zone, the energy-transfer service can use a Total Site target as its base. The returned target keeps the usual utility summary fields and adds the interval table plus a graph-neutral diagram data.


In [ ]:
site_problem = PinchProblem(local_case_path, project_name="pulp_mill_energy_transfer")
site_energy_transfer = site_problem.target.energy_transfer(
    options={"base_target_type": "Total Site Target"},
)

{
    "target_name": site_energy_transfer.name,
    "base_target_type": site_energy_transfer.base_target_type,
    "base_target_name": site_energy_transfer.base_target_name,
    "table_rows": len(site_energy_transfer.heat_surplus_deficit_table),
    "operation_profiles": len(site_energy_transfer.energy_transfer_diagram["operations"]),
}


## Heat-surplus/deficit table
The table is interval-based. Each row records a shared ETD temperature point, each operation's interval `Delta Hnet`, and the net surplus or deficit for that interval.


In [ ]:
surplus_deficit = pd.DataFrame(site_energy_transfer.heat_surplus_deficit_table)
surplus_deficit.head(10)


## Energy transfer diagram data
The diagram data stores operation-level cascades on a shared temperature grid. The plot uses each operation's stacked heat profile, while custom reports can inspect the interval heat and cascade heat arrays directly.


In [ ]:
diagram = site_energy_transfer.energy_transfer_diagram

pd.DataFrame(
    [
        {
            "operation": operation["name"],
            "mode": operation["mode"],
            "max_heat": operation["max_heat"],
            "cross_pinch": operation["cross_pinch"],
            "points": len(operation["stacked_heat"]),
        }
        for operation in diagram["operations"]
    ]
)


## Graph catalog and plot accessor
The same target also appears in the standard graph catalog as `Energy Transfer Diagram`, so it can be selected or exported through the normal plot accessor.


In [ ]:
catalog = site_problem.plot.catalog()
catalog[catalog["Graph Type"].eq("Energy Transfer Diagram")]


In [ ]:
site_problem.plot.energy_transfer_diagram()

In [ ]:
energy_transfer_graph = site_problem.plot.energy_transfer_diagram(
    return_graph_data=True,
)

{
    "graph_type": energy_transfer_graph["type"],
    "segments": len(energy_transfer_graph["segments"]),
    "first_segment": energy_transfer_graph["segments"][0]["title"],
}


## Local direct-integration view
The same service can use Direct Integration as the base target. This is useful when you want the energy-transfer picture for one process zone instead of the aggregate Total Site scope.


In [ ]:
bleaching_energy_transfer = site_problem.target.energy_transfer(
    zone_name="Bleaching",
    options={"base_target_type": "Direct Integration"},
)

pd.DataFrame(
    [
        {
            "scope": "Total Site",
            "base": site_energy_transfer.base_target_type,
            "Qh": site_energy_transfer.hot_utility_target,
            "Qc": site_energy_transfer.cold_utility_target,
            "rows": len(site_energy_transfer.heat_surplus_deficit_table),
        },
        {
            "scope": "Bleaching",
            "base": bleaching_energy_transfer.base_target_type,
            "Qh": bleaching_energy_transfer.hot_utility_target,
            "Qc": bleaching_energy_transfer.cold_utility_target,
            "rows": len(bleaching_energy_transfer.heat_surplus_deficit_table),
        },
    ]
)


## Interpretation pattern
Use the summary target fields for utility magnitude, the surplus/deficit table for interval accounting, and the diagram data or `plot.energy_transfer_diagram(...)` when a reporting layer needs a visual transfer path.
